# 2.3 搬运指令：数据的搬运工家族

上一节我们说“搬运是宿命”——本节认识专职搬运工：**MTE2 / MTE1 / MTE3 / FIXPIPE** 四兄弟，外加 950 新晋的 **NDDMA**。这是第三块积木，FA 性能的第一战场（搬运与计算的重叠程度，直接决定 Kernel 快慢）。

## 1. 四类搬运指令与存储层级的对应

<img src="./images/mte_storage_map.svg" width="720">

> **看图要点**：
> - **MTE2**：片外 GM → 片上（L1 / UB / L0A/B），大块数据搬运主力；搬往 L1 时按分形大小（512B）对齐搬运性能更优；
> - **MTE1**：L1 → L0A / L0B（分离架构下还负责 L1 → FP/BT Buffer），专供 Cube 的输入，**可顺带做 ND → NZ 分形转换**；
> - **MTE3**：UB → GM、UB → L1、L1 → GM，结果输出通道（UB→GM 是 Vector 结果回写）；
> - **FIXPIPE**：L0C → GM / UB 的专用通路，**可随路完成量化与排布转换**（950 的随路量化即走此路）。

| 指令 | 路径 | 一句话职责 | FA 中的典型任务 |
|--|--|--|--|
| MTE2 | GM → L1/UB/L0 | 大块输入搬运 | Q/K/V 块搬入片上 |
| MTE1 | L1 → L0A/L0B（及 FP/BT Buffer） | Cube 输入 + 分形 | K/V 进矩阵乘前就位 |
| MTE3 | UB → GM/L1、L1 → GM | 结果输出 | O 写回 GM |
| FIXPIPE | L0C → GM/UB | 定点结果 + 随路量化 | 950 量化 FA 的输出；S 直入 UB |

> **关键认知**：各 MTE 有**独立的指令队列**，与 Cube/Vector 计算并行——这就是“搬运与计算重叠”的硬件基础，也是下一小节流水线的前提。

> **带宽量级直觉**（字节/周期，910B → 910D）：MTE2 恒为 256B/c；MTE1 对 L0A 为 256B/c，对 L0B 从 128B/c 提升到 256B/c；MTE3 仅 128B/c（每 Vector 核）；910D 新增的 L1↔UB 与 L0C→UB 直连为 256B/c。要点有二：**搬出天生比搬入“贵”**（MTE3 半宽），且**左右矩阵的待遇不同**（L0B 曾是半宽）——这些不对等都会在第7章搬运占比分析中现形。

## 2. NDDMA（950）：搬运 + 5 维重排一步到位

950 新增的 NDDMA 引擎，把传统“搬运后再整理”合并为**单条指令**：

<img src="./images/nddma_5d.png" width="560">

> **看图要点**：NDDMA 在搬运的同时完成最多 **5 个维度**的数据重排（transpose/reshape 等）。对 FA 的意义：GQA 场景下 K/V 的头维度重排、BSND ↔ BNSD 布局调整，都可以“免费”搭在搬运里完成，省去额外的片上整理开销。

## 3. 搬运与计算的重叠：流水线并行

搬运和计算如果串行执行，一半时间硬件在“等货”：

<img src="./images/pipeline_parallelism.png" width="600">

> **看图要点**：把任务切成 T1、T2、T3 多轮后，**下一轮的搬运与上一轮的计算重叠**——MTE 在搬第 2 块时，Cube/Vector 在算第 1 块。理想状态下，总耗时 ≈ max(总搬运时间, 总计算时间)，而非两者之和。FA 的 KV 块天然适合这种切法（第3篇 FA Kernel 的主循环就是它）。

## 4. Double Buffering：流水线的标准实现

流水线最常用的工程手法——给每类数据开**两份缓冲**，一份在算、一份在搬：

<img src="./images/double_buffer.png" width="600">

> **看图要点**：Buffer 0 在被计算时，MTE2 正往 Buffer 1 写数据；下一轮两者对调。指令队列独立使这种重叠不需要程序员手动插同步（队列机制见 2.6 节）。在 Ascend C 中只需 `InitBuffer(que, 2, size)`（缓冲份数 = 2）即可开启，FA Kernel 的第 5.3 节“buffer 复用（开 pingpong）”正是这个机制的实战。

> **代价**：双缓冲使每类数据的片上占用 ×2——所以 Tiling 设计（容量预算）与流水线设计（重叠收益）必须联合权衡。这个矛盾贯穿第3篇。

## 小结

| 关键词 | 一句话 |
|--|--|
| 四兄弟 | MTE2（进）、MTE1（Cube 口）、MTE3（出）、FIXPIPE（量化出） |
| NDDMA | 950 新增，搬运 + 5 维重排单指令 |
| 流水线 | 搬运与计算重叠，耗时取 max 而非和 |
| 双缓冲 | 两份缓冲轮转，是流水线的标准实现 |

下一节 [2.4 Cube 计算指令](01.06_cube_instructions.ipynb)：货物就位后，Cube 怎么干活。

## 课后练习

1.（单选）把 Q 矩阵从 GM 搬到 L1，应使用哪条搬运指令？（A. MTE1　B. MTE2　C. MTE3　D. FIXPIPE）

2.（判断）开启 Double Buffering 后，每类数据的片上内存占用会翻倍。

3.（单选）950 上想在搬运 K/V 的同时完成 GQA 头维度的重排，最合适的搬运单元是？（A. MTE2　B. MTE1　C. NDDMA　D. FIXPIPE）

> 完成后查看 [answer/01.05_answer.txt](answer/01.05_answer.txt) 核对答案。